# Clone GEE Image Collections between projects

Copies image collections from the following source folders into `projects/mangrove-atlas-246414/assets/`:

| Source folder | Destination folder |
|---|---|
| `projects/global-mangrove-watch/land-cover/` | `…/assets/land-cover` |
| `projects/global-mangrove-watch/mangrove-properties/` | `…/assets/mangrove-properties` |
| `projects/global-mangrove-watch/physical-environment/` | `…/assets/physical-environment` |

Steps:
1. List all image collections in the source folder.
2. For each collection, create a matching image collection in the destination.
3. List all images inside the source collection.
4. Copy each image asset to the destination collection.

> **Requirements**: You must have read access to the source project and write access to the destination project.

## Setup

In [1]:
import ee
import logging
from typing import List, Dict, Optional

logging.basicConfig(level=logging.INFO)

In [ ]:
# Authenticate and initialise Earth Engine
ee.Authenticate()
ee.Initialize()

## Configuration

In [20]:
SOURCE_FOLDER = 'projects/global-mangrove-watch/land-cover'
DEST_FOLDER   = 'projects/mangrove-atlas-246414/assets/land-cover'

# Set to a list of collection names (without path) to restrict which collections
# are copied. Leave as None to copy everything found in SOURCE_FOLDER.
# Example: COLLECTION_FILTER = ['mangrove_extent-v3', 'mangrove_height-v3']
COLLECTION_FILTER: Optional[List[str]] = None

# Set to True to overwrite assets that already exist in the destination.
ALLOW_OVERWRITE = False

## Helper functions

In [21]:
def list_all_assets(parent: str) -> List[Dict]:
    """List every asset under `parent`, handling pagination automatically."""
    assets = []
    page_token = None
    while True:
        params = {'parent': parent}
        if page_token:
            params['pageToken'] = page_token
        response = ee.data.listAssets(params)
        assets.extend(response.get('assets', []))
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    return assets

In [22]:
def asset_name(asset_id: str) -> str:
    """Return the last path component of an asset id."""
    return asset_id.split('/')[-1]


def dest_id(source_name: str, source_parent: str, dest_parent: str) -> str:
    """Rebase `source_name` from `source_parent` to `dest_parent`.

    Asset names returned by the API may carry a legacy prefix
    (``projects/earthengine-legacy/assets/``).  Strip it first so the
    rebasing works for any source project path.
    """
    canonical = source_name.removeprefix('projects/earthengine-legacy/assets/')
    relative = canonical[len(source_parent):].lstrip('/')
    return f'{dest_parent}/{relative}'

In [23]:
def ensure_image_collection(path: str, source_properties: Dict) -> None:
    """Create an ImageCollection at `path` if it does not already exist.

    Copies the top-level properties from the source collection so that
    metadata (name, keywords, etc.) is preserved.
    """
    try:
        ee.data.getAsset(path)
        logging.info(f'Collection already exists, skipping creation: {path}')
    except Exception:
        # getAsset raises when the asset does not exist
        properties = source_properties.get('properties', {})
        ee.data.createAsset({'type': 'ImageCollection', **({'properties': properties} if properties else {})}, path)
        logging.info(f'Created image collection: {path}')

In [24]:
def copy_image(source_id: str, dest_image_id: str, allow_overwrite: bool = False) -> None:
    """Copy a single image asset from `source_id` to `dest_image_id`."""
    try:
        ee.data.copyAsset(source_id, dest_image_id, allowOverwrite=allow_overwrite)
        logging.info(f'  Copied: {source_id} -> {dest_image_id}')
    except ee.EEException as exc:
        if 'already exists' in str(exc):
            logging.warning(f'  Already exists (skipped): {dest_image_id}')
        else:
            logging.error(f'  Failed to copy {source_id}: {exc}')
            raise

In [25]:
# Asset types that are copied as-is (not recursed into)
DIRECT_COPY_TYPES = {'IMAGE', 'TABLE', 'FEATURE_VIEW', 'CLASSIFIER', 'ARRAY'}


def clone_folder(
    source_folder: str,
    dest_folder: str,
    collection_filter: Optional[List[str]] = None,
    allow_overwrite: bool = False,
) -> None:
    """Clone every ImageCollection and direct asset under *source_folder* into *dest_folder*.

    - ``IMAGE_COLLECTION``: the collection is recreated and all contained images are copied.
    - ``IMAGE``, ``TABLE``, and other direct types: copied straight to *dest_folder*.

    Args:
        source_folder: GEE asset path of the source folder.
        dest_folder: GEE asset path of the destination folder.
        collection_filter: If given, only assets whose bare name is in this list are
            processed.  ``None`` processes everything.
        allow_overwrite: Passed through to copy calls.
    """
    all_assets = list_all_assets(source_folder)
    collections   = [a for a in all_assets if a.get('type') == 'IMAGE_COLLECTION']
    direct_assets = [a for a in all_assets if a.get('type') in DIRECT_COPY_TYPES]

    if collection_filter is not None:
        collections   = [c for c in collections   if asset_name(c['name']) in collection_filter]
        direct_assets = [a for a in direct_assets if asset_name(a['name']) in collection_filter]

    print(f'Found in {source_folder}:')
    print(f'  {len(collections)} image collection(s)')
    print(f'  {len(direct_assets)} direct asset(s)')

    # -- image collections -------------------------------------------------------
    for collection in collections:
        src_col = collection['name']
        dst_col = dest_id(src_col, source_folder, dest_folder)

        logging.info(f'\n=== Collection: {src_col} ===')
        source_meta = ee.data.getAsset(src_col)
        ensure_image_collection(dst_col, source_meta)

        images = list_all_assets(src_col)
        logging.info(f'Copying {len(images)} image(s)...')
        for img in images:
            dst_img = dest_id(img['name'], source_folder, dest_folder)
            copy_image(img['name'], dst_img, allow_overwrite)

        logging.info(f'Done: {dst_col}')

    # -- direct assets -----------------------------------------------------------
    for asset in direct_assets:
        src_asset = asset['name']
        dst_asset = dest_id(src_asset, source_folder, dest_folder)

        logging.info(f'\n=== Asset [{asset["type"]}]: {src_asset} ===')
        copy_image(src_asset, dst_asset, allow_overwrite)   # copy_image works for any asset type
        logging.info(f'Done: {dst_asset}')

    # -- verify ------------------------------------------------------------------
    result = list_all_assets(dest_folder)
    result_cols   = [a for a in result if a.get('type') == 'IMAGE_COLLECTION']
    result_direct = [a for a in result if a.get('type') in DIRECT_COPY_TYPES]

    print(f'\nDestination {dest_folder} now has:')
    print(f'  {len(result_cols)} collection(s):')
    for c in result_cols:
        imgs = list_all_assets(c['name'])
        print(f'    {c["name"]}  ({len(imgs)} image(s))')
    print(f'  {len(result_direct)} direct asset(s):')
    for a in result_direct:
        print(f'    {a["name"]}  [{a["type"]}]')

## 1 — List source image collections

In [13]:
all_source_assets = list_all_assets(SOURCE_FOLDER)
source_collections = [
    a for a in all_source_assets
    if a.get('type') == 'IMAGE_COLLECTION'
]

if COLLECTION_FILTER is not None:
    source_collections = [
        c for c in source_collections
        if asset_name(c['name']) in COLLECTION_FILTER
    ]

print(f'Found {len(source_collections)} image collection(s) to copy:')
for c in source_collections:
    print(f'  {c["name"]}')

Found 12 image collection(s) to copy:
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent-distance_version-2-0_1996--2016
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent-gain
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent-gain_version-2-0_1996--2016
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent-loss
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent-loss_version-2-0_1996--2016
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove-extent_version-2-0_1996--2016
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover

In [15]:
# Keep only v3 assets
source_collections = [c for c in source_collections    if 'v3' in asset_name(c['name'])]
source_collections

[{'type': 'IMAGE_COLLECTION',
  'name': 'projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3',
  'id': 'projects/global-mangrove-watch/land-cover/mangrove_extent-v3',
  'updateTime': '2022-08-10T14:39:17.414305Z'},
 {'type': 'IMAGE_COLLECTION',
  'name': 'projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent_gain-v3',
  'id': 'projects/global-mangrove-watch/land-cover/mangrove_extent_gain-v3',
  'updateTime': '2022-08-11T14:50:33.135199Z'},
 {'type': 'IMAGE_COLLECTION',
  'name': 'projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent_loss-v3',
  'id': 'projects/global-mangrove-watch/land-cover/mangrove_extent_loss-v3',
  'updateTime': '2022-08-11T15:23:02.437679Z'}]

## 2 — Preview destination paths

Review this before running the copy cell.

In [18]:
for c in source_collections:
    src = c['name']
    dst = dest_id(src, SOURCE_FOLDER, DEST_FOLDER)
    images = list_all_assets(src)
    print(f'{src}  ({len(images)} image(s))')
    print(f'  -> {dst}')

projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3  (11 image(s))
  -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3
projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent_gain-v3  (10 image(s))
  -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent_gain-v3
projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent_loss-v3  (10 image(s))
  -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent_loss-v3


## 3 — Copy collections

For each source collection:
- Creates the destination `ImageCollection` (preserving collection-level properties).
- Copies every image inside it.

In [20]:
for collection in source_collections:
    src_collection_id = collection['name']
    dst_collection_id = dest_id(src_collection_id, SOURCE_FOLDER, DEST_FOLDER)

    logging.info(f'\n=== Processing collection: {src_collection_id} ===')

    # Fetch source collection metadata to preserve properties
    source_meta = ee.data.getAsset(src_collection_id)

    # Create destination collection if needed
    ensure_image_collection(dst_collection_id, source_meta)

    # List and copy each image
    images = list_all_assets(src_collection_id)
    logging.info(f'Copying {len(images)} image(s)...')

    for img in images:
        src_img_id = img['name']
        dst_img_id = dest_id(src_img_id, SOURCE_FOLDER, DEST_FOLDER)
        copy_image(src_img_id, dst_img_id, allow_overwrite=ALLOW_OVERWRITE)

    logging.info(f'Done: {dst_collection_id}')

INFO:root:
=== Processing collection: projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3 ===
INFO:root:Created image collection: projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3
INFO:root:Copying 11 image(s)...
INFO:root:  Copied: projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3/gmw_v3_1996 -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3/gmw_v3_1996
INFO:root:  Copied: projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3/gmw_v3_2007 -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3/gmw_v3_2007
INFO:root:  Copied: projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-cover/mangrove_extent-v3/gmw_v3_2008 -> projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3/gmw_v3_2008
INFO:root:  Copied: projects/earthengine-legacy/assets/projects/global-mangrove-watch/land-c

## 4 — Verify

List what ended up in the destination folder.

In [24]:
dest_assets = list_all_assets(DEST_FOLDER)
dest_collections = [a for a in dest_assets if a.get('type') == 'IMAGE_COLLECTION']

print(f'Destination folder contains {len(dest_collections)} image collection(s):')
for c in dest_collections:
    images = list_all_assets(c['name'])
    print(f'  {c["name"]}  ({len(images)} image(s))')

Destination folder contains 3 image collection(s):
  projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent-v3  (11 image(s))
  projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent_gain-v3  (10 image(s))
  projects/mangrove-atlas-246414/assets/land-cover/mangrove_extent_loss-v3  (10 image(s))


---
## mangrove-properties

Clones image collections from `projects/global-mangrove-watch/mangrove-properties/` into `projects/mangrove-atlas-246414/assets/mangrove-properties`.

In [15]:
MP_SOURCE = 'projects/global-mangrove-watch/mangrove-properties'
MP_DEST   = 'projects/mangrove-atlas-246414/assets/mangrove-properties'

# Restrict to specific collections, or None to copy all.
MP_FILTER: Optional[List[str]] = None

# Preview what will be cloned
all_mp = list_all_assets(MP_SOURCE)
mp_collections = [a for a in all_mp if a.get('type') == 'IMAGE_COLLECTION']
if MP_FILTER:
    mp_collections = [c for c in mp_collections if asset_name(c['name']) in MP_FILTER]

print(f'Found {len(mp_collections)} collection(s) to clone:')
for c in mp_collections:
    print(f'  {c["name"]}  ->  {dest_id(c["name"], MP_SOURCE, MP_DEST)}')

Found 7 collection(s) to clone:
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/mangrove-properties/mangrove_aboveground_biomass-v3  ->  projects/mangrove-atlas-246414/assets/mangrove-properties/mangrove_aboveground_biomass-v3
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/mangrove-properties/mangrove_aboveground_biomass_1996-2016  ->  projects/mangrove-atlas-246414/assets/mangrove-properties/mangrove_aboveground_biomass_1996-2016
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/mangrove-properties/mangrove_basal-area_weighted_height_2000_v1-2  ->  projects/mangrove-atlas-246414/assets/mangrove-properties/mangrove_basal-area_weighted_height_2000_v1-2
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/mangrove-properties/mangrove_blue_carbon-v2_1  ->  projects/mangrove-atlas-246414/assets/mangrove-properties/mangrove_blue_carbon-v2_1
  projects/earthengine-legacy/assets/projects/global-mangrove-watch/mangrove-p

In [ ]:
clone_folder(MP_SOURCE, MP_DEST, collection_filter=MP_FILTER, allow_overwrite=False)

---
## physical-environment

Clones image collections from `projects/global-mangrove-watch/physical-environment/` into `projects/mangrove-atlas-246414/assets/physical-environment`.

In [27]:
PE_SOURCE = 'projects/global-mangrove-watch/physical-environment'
PE_DEST   = 'projects/mangrove-atlas-246414/assets/physical-environment'

# Restrict to specific asset names, or None to copy all.
PE_FILTER: Optional[List[str]] = ['coastlines-v3']

# Preview what will be cloned
all_pe      = list_all_assets(PE_SOURCE)
pe_cols     = [a for a in all_pe if a.get('type') == 'IMAGE_COLLECTION']
pe_direct   = [a for a in all_pe if a.get('type') in DIRECT_COPY_TYPES]

if PE_FILTER:
    pe_cols   = [c for c in pe_cols   if asset_name(c['name']) in PE_FILTER]
    pe_direct = [a for a in pe_direct if asset_name(a['name']) in PE_FILTER]

print(f'Image collections ({len(pe_cols)}):')
for c in pe_cols:
    print(f'  {c["name"]}  ->  {dest_id(c["name"], PE_SOURCE, PE_DEST)}')

print(f'\nDirect assets ({len(pe_direct)}):')
for a in pe_direct:
    print(f'  [{a["type"]}]  {a["name"]}  ->  {dest_id(a["name"], PE_SOURCE, PE_DEST)}')

Image collections (0):

Direct assets (1):
  [TABLE]  projects/earthengine-legacy/assets/projects/global-mangrove-watch/physical-environment/coastlines-v3  ->  projects/mangrove-atlas-246414/assets/physical-environment/coastlines-v3


In [28]:
clone_folder(PE_SOURCE, PE_DEST, collection_filter=PE_FILTER, allow_overwrite=False)

INFO:root:
=== Asset [TABLE]: projects/earthengine-legacy/assets/projects/global-mangrove-watch/physical-environment/coastlines-v3 ===


Found in projects/global-mangrove-watch/physical-environment:
  0 image collection(s)
  1 direct asset(s)


INFO:root:  Copied: projects/earthengine-legacy/assets/projects/global-mangrove-watch/physical-environment/coastlines-v3 -> projects/mangrove-atlas-246414/assets/physical-environment/coastlines-v3
INFO:root:Done: projects/mangrove-atlas-246414/assets/physical-environment/coastlines-v3



Destination projects/mangrove-atlas-246414/assets/physical-environment now has:
  0 collection(s):
  1 direct asset(s):
    projects/mangrove-atlas-246414/assets/physical-environment/coastlines-v3  [TABLE]
